# VRC-3PO revision — R1d: published detectors re-implemented (Reviewer 1, Comment 3)

**Question:** how does a published cybersickness detector behave on its own source data under (a) the protocol it was published with and (b) the benchmark's participant-held-out, environment-controlled protocol?

| model | source | what is rebuilt |
|---|---|---|
| `litevr:LSTM` | LiteVR (Kundu et al. 2023), Table 8 | LSTM-64 (dropout 0.15) → Dense-32 → sigmoid |
| `litevr:GRU` | LiteVR, Table 9 | GRU-32 → GRU-64 (dropout 0.2) → Dense-32 → sigmoid |
| `litevr:MLP` | LiteVR, Table 10 | Dense 64 → 32 → 16 (dropout 0.25) → sigmoid |
| `feat:ebm` | TruVR (Kundu et al. 2022) | Explainable Boosting Machine, lr 0.001, early-stopping patience 30, on window statistics |

LiteVR was trained on the Islam et al. recordings — the benchmark's **Simulations** source folder. Adaptations (stated in the paper): 30 × 14 windows at 1 Hz, binary sigmoid output, flattened window for the MLP, the benchmark's shared training recipe.

| protocol | flag | meaning |
|---|---|---|
| theirs | `--protocol windows` | 10-fold over **windows**; the same participants (and overlapping windows) appear in training and test |
| ours | `--protocol cv` | repeated participant-level 5-fold; test participants never seen in training |

Every result is reported with the environment-only baseline, the increment over it, and the within-environment AUC.

**Runtime on a T4:** ≈ 2–3 h in total, resumable (each fold is one file on Drive). Steps 2 and 3 are the essential ones; 4 and 5 add the Simulations-only participant CV and the seated sample-level run.

In [ ]:
# ═══ 1 · Runtime ═══
!pip install -q keras-tcn interpret-core
import tensorflow as tf, pandas as pd, numpy as np
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ═══ 2 · Drive + code (bundle v4) ═══
from google.colab import drive, files
from pathlib import Path
import zipfile, shutil, os
drive.mount('/content/drive')
REV = Path('/content/drive/MyDrive/vrc3po_revision')
BUNDLE = REV / 'vrc3po_revision_bundle_v4.zip'
if not BUNDLE.exists():
    print('Upload vrc3po_revision_bundle_v4.zip from the VRC-3PO folder on your Mac:')
    up = files.upload()
    shutil.copy([n for n in up if n.endswith('.zip')][0], BUNDLE)
root = Path('/content/vrc3po_code'); shutil.rmtree(root, ignore_errors=True); root.mkdir()
zipfile.ZipFile(BUNDLE).extractall(root); os.chdir(root)
assert (root / 'analysis/litevr.py').exists(), 'wrong bundle: needs v4'
!pip install -q pytest
!python -m pytest tests -q

In [ ]:
# ═══ 3 · Paths; reuse the published-recipe ensemble and baselines for pairing ═══
V11 = REV / 'vrc3po_benchmark_v1_1.csv'
OUT = REV / 'litevr'                       # separate store: summaries stay fast
assert V11.exists(), V11
(OUT / 'runs').mkdir(parents=True, exist_ok=True)
copied = 0
for pattern in ['cnn_ensemble__fms_gt2__seated__cv__*.npz',
                'nuisance_condition__fms_gt2__seated__cv__*.npz',
                'nuisance_condition_clock__fms_gt2__seated__cv__*.npz']:
    for f in (REV / 'cv' / 'runs').glob(pattern):
        if not (OUT / 'runs' / f.name).exists():
            shutil.copy(f, OUT / 'runs' / f.name); copied += 1
print('copied', copied, 'reference runs; store now has', len(list((OUT / 'runs').glob('*.npz'))), 'runs')
LITEVR = ['litevr:LSTM', 'litevr:GRU', 'litevr:MLP']
BASE = ['nuisance:condition', 'nuisance:condition_clock']

## Step 1 · Smoke test (2 folds, 2 epochs, ~2 min)

In [ ]:
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$REV/smoke_r1d" --scope simulations --protocol windows --models litevr:LSTM litevr:GRU litevr:MLP feat:ebm --repeats 1 --folds 2 --epochs 2
shutil.rmtree(REV / 'smoke_r1d', ignore_errors=True)
print('Smoke test passed.')

## Step 2 · Their protocol on their data (Simulations, 10-fold over windows) · ≈ 30–45 min
This is the setting LiteVR reported 94% accuracy in.

In [ ]:
assert tf.config.list_physical_devices('GPU'), 'Switch to a GPU runtime.'
MODELS = ' '.join(LITEVR + ['feat:ebm'] + BASE)
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol windows --folds 10 --repeats 1 --models {MODELS}

## Step 3 · Our protocol (47 seated participants, 5×5 participant CV) · ≈ 60–90 min
Identical folds to the published-recipe ensemble, so the comparison is paired.

In [ ]:
MODELS = ' '.join(LITEVR + ['feat:ebm'])
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope seated --protocol cv --folds 5 --repeats 5 --models {MODELS}

## Step 4 · Our protocol on their data (Simulations only, 3×5 participant CV) · ≈ 30–45 min
Same windows as Step 2, so the protocol effect is measured on identical data.

In [ ]:
MODELS = ' '.join(LITEVR + ['feat:ebm'] + BASE)
!python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope simulations --protocol cv --folds 5 --repeats 3 --models {MODELS}

## Step 5 · (optional) Their protocol on the full seated subset · ≈ 30–45 min

In [ ]:
RUN_STEP5 = True
if RUN_STEP5:
    MODELS = ' '.join(LITEVR + ['feat:ebm'] + BASE)
    !python -m analysis.revision_cv run --dataset "$V11" --output-dir "$OUT" --scope seated --protocol windows --folds 10 --repeats 1 --models {MODELS}

## Step 6 · Summary → `results_revision_r1d.zip`

In [ ]:
import subprocess
proc = subprocess.Popen(['python', '-m', 'analysis.revision_cv', 'summarize', '--dataset', str(V11),
                         '--output-dir', str(OUT), '--bootstrap', '2000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith('['): print(line, end='')
assert proc.wait() == 0, 'summary failed'
S = OUT / 'summary'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
p = pd.read_csv(S / 'primary_estimates.csv'); p = p[p.label == 'fms_gt2']
e = pd.read_csv(S / 'environment_control.csv'); e = e[e.label == 'fms_gt2']
t = p.merge(e[['scope', 'protocol', 'model', 'auc_environment_only', 'increment', 'increment_ci_low',
               'increment_ci_high', 'increment_over_env_clock', 'increment_env_clock_ci_low',
               'increment_env_clock_ci_high']], on=['scope', 'protocol', 'model'], how='left')
cols = ['scope', 'protocol', 'model', 'repeats', 'windows', 'auc_mean', 'auc_ci_low', 'auc_ci_high',
        'within_condition_auc', 'within_condition_ci_low', 'within_condition_ci_high',
        'auc_environment_only', 'increment', 'increment_ci_low', 'increment_ci_high',
        'increment_over_env_clock', 'increment_env_clock_ci_low', 'increment_env_clock_ci_high']
order = {'windows': 0, 'cv': 1}
t = t.assign(_o=t.protocol.map(order)).sort_values(['scope', '_o', 'model'])[cols]
display(t.round(3))
op = pd.read_csv(S / 'operating_points_validation_selected.csv')
op = op[(op.operating_point == 'default_0.5') & op.model.str.startswith(('litevr', 'feat:ebm'))]
op['balanced_accuracy'] = (op.sensitivity + op.specificity) / 2
print('\nDefault threshold (0.5): sensitivity, specificity, balanced accuracy')
display(op[['scope', 'protocol', 'model', 'sensitivity', 'specificity', 'balanced_accuracy']].round(3))
t.round(4).to_csv(S / 'r1d_table.csv', index=False)
zip_path = shutil.make_archive(str(REV / 'results_revision_r1d'), 'zip', S)
print('saved', zip_path)
files.download(zip_path)

**What to send back:** the `results_revision_r1d.zip` (or paste the two tables above).

Reading guide: for each model, compare the `windows` row with the `cv` row on the same scope (protocol effect), and the `cv` row with `auc_environment_only` and `within_condition_auc` (environment effect).